# Eddy Stage 1 Loading and QC

This tutorial loads flux and biomet CSV fixtures through `miaproc.eddy.load_stage1`. Stage 1 combines files, creates timestamps, applies QC/rain filtering, and regularizes to a 30-minute grid.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "mia-tutorials" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA = ROOT / "data"
REPOS = ROOT.parent
ROOT

import pandas as pd
from miaproc.eddy import load_stage1

eddy_dir = DATA / "miaproc_eddy"
flux_dir = eddy_dir / "full_output"
biomet_dir = eddy_dir / "biomet"
flux_dir, biomet_dir


## Inspect raw columns

The source fixture mirrors EddyPro-style full output plus a biomet table.


In [ ]:
flux_preview = pd.read_csv(flux_dir / "full_output.csv", nrows=3)
biomet_preview = pd.read_csv(biomet_dir / "biomet.csv", nrows=3)

print("flux columns", len(flux_preview.columns))
print(flux_preview[["date", "time", "co2_flux", "u.", "air_temperature", "VPD"]].head())
print("\nbiomet columns", len(biomet_preview.columns))
print(biomet_preview[["date", "time", "SWIN_1_1_1", "P_RAIN_1_1_1", "RH_1_1_1"]].head())


## Load stage 1

The fixture has no units row, so both `skip_*` values are zero. Rain rows are dropped here to mirror the package tests.


In [ ]:
stage1 = load_stage1(
    path_full_output=flux_dir,
    path_biomet=biomet_dir,
    tz_in="UTC",
    tz_out="UTC",
    skip_full_output=0,
    skip_biomet=0,
    drop_rain_rows=True,
)

stage1.shape, stage1["DateTime"].min(), stage1["DateTime"].max()


In [ ]:
core_cols = ["DateTime", "NEE", "USTAR", "QC_NEE", "Tair", "VPD", "Rg", "rH", "P_RAIN"]
stage1[core_cols].head()


## Check the regularized time grid

A successful stage-1 frame should be strictly 30-minute spaced after regularization.


In [ ]:
diffs = stage1["DateTime"].sort_values().diff().dropna()
print(diffs.value_counts().head())
assert (diffs == pd.Timedelta(minutes=30)).all()


In [ ]:
stage1[core_cols].describe().T[["count", "mean", "min", "max"]]
